# Training a Support Vector Classifier with a Quantum Evolution Kernel — Student notebook


This notebook follows the lab handout and provides the **code skeleton** needed to complete it within the session.

The task is modeled on the molecular-toxicity experiment of Albrecht *et al.* An input graph represents a **molecule**: vertices are atoms and edges are chemical bonds. The binary labels are

- `y = 0`: **non-toxic molecule**;
- `y = 1`: **toxic molecule**.

As in the previous lab, each graph is represented after the quantum evolution by an excitation-count probability distribution

$$P_G = \big(P(N_R=0),P(N_R=1),\ldots\big)$$

For this practical, the costly neutral-atom feature-extraction stage is **not rerun**. The helper cell below generates a small synthetic surrogate dataset of such distributions. Each row of `P` should be interpreted as the quantum feature associated with one molecular graph. The graph size is retained in `sizes`.

Cells marked **TODO** are the ones you are expected to complete.


### Setup
Run the following cell once.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import binom
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    balanced_accuracy_score,
)

np.set_printoptions(precision=4, suppress=True)

### Provided dataset cell
This cell is **not a question**, do not modify it. It generates a small reproducible **surrogate** for the excitation-count distributions of a binary molecular-toxicity dataset.

Each sample represents one hypothetical molecular graph after the quantum feature-extraction stage:
- `P[i]` is its normalized excitation-count distribution on the common support `0,...,12`;
- `y[i] = 0` means **non-toxic**, while `y[i] = 1` means **toxic**;
- `sizes[i]` is the number of vertices (atoms) in the corresponding graph.

The graph adjacency itself is not generated in this helper cell. The purpose is to start directly from the quantum features, as if they had already been measured on a QPU or emulator.



In [ ]:
def make_excitation_dataset(seed=7, n_per_class=80, shots=500, n_max=12):
    rng = np.random.default_rng(seed)
    P, y, sizes = [], [], []

    # cls = 0: non-toxic surrogate; cls = 1: toxic surrogate
    for cls in [0, 1]:
        for _ in range(n_per_class):
            if cls == 0:
                N = int(rng.choice(np.arange(6, 12), p=[.22, .20, .19, .16, .13, .10]))
                a = np.clip(rng.normal(.17, .04), .04, .35)
                b = np.clip(rng.normal(.35, .05), .12, .60)
                w = np.clip(rng.normal(.60, .12), .25, .90)
            else:
                N = int(rng.choice(np.arange(7, 13), p=[.10, .13, .17, .19, .20, .21]))
                a = np.clip(rng.normal(.20, .04), .05, .38)
                b = np.clip(rng.normal(.40, .05), .15, .65)
                w = np.clip(rng.normal(.46, .12), .15, .80)

            k = np.arange(N + 1)
            probs = w * binom.pmf(k, N, a) + (1 - w) * binom.pmf(k, N, b)
            probs /= probs.sum()

            counts = rng.multinomial(shots, probs)
            p = np.zeros(n_max + 1)
            p[:N + 1] = counts / shots

            P.append(p)
            y.append(cls)
            sizes.append(N)

    return np.asarray(P), np.asarray(y), np.asarray(sizes)

P, y, sizes = make_excitation_dataset()
P.shape, y.shape, sizes.shape

## Q1 — Dataset preparation

Report:
- the number of samples;
- the dimension of each probability-distribution vector;
- the number of samples in each class;
- the range of graph sizes.

Then create a **stratified 75/25 train/test split** with `random_state=42`.

Keep the test set untouched until the final evaluation.


In [ ]:
# TODO — inspect the dataset
print("Number of samples:", ...)
print("Feature dimension:", ...)
print("Class counts:", ...)
print("Graph-size range:", ...)

# TODO — build a stratified 75/25 split
idx = np.arange(len(y))

idx_train, idx_test = train_test_split(
    idx,
    test_size=...,       # TODO
    stratify=...,        # TODO
    random_state=42,
)

# TODO — construct train/test arrays
P_train, P_test = ..., ...
y_train, y_test = ..., ...
sizes_train, sizes_test = ..., ...

print("Train shape:", P_train.shape)
print("Test shape :", P_test.shape)
print("Train class counts:", np.bincount(y_train))
print("Test class counts :", np.bincount(y_test))

### Jensen–Shannon divergence — provided

As in the previous lab,

$$\mathrm{JSD}(P,Q)
=
H\left(\frac{P+Q}{2}\right)
-
\frac{H(P)+H(Q)}{2},
\qquad
H(P)=-\sum_i P_i\log_2 P_i.$$

The implementation is given below.


In [ ]:
def js_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def entropy(a):
        mask = a > 0
        return -np.sum(a[mask] * np.log2(a[mask]))

    return entropy(m) - 0.5 * (entropy(p) + entropy(q))

## Q2 — QEK similarity and the role of the hyperparameter $\mu$

Use

$$K_\mu(P,Q)=\exp[-\mu\,\mathrm{JSD}(P,Q)].$$

Complete the function below, then study how the similarity between two different graph signatures changes with \(\mu\).


In [ ]:
def qek_similarity(p, q, mu=1.0):
    # TODO — implement K_mu(P,Q)
    return ...


# Pick one training graph and one graph from the opposite class.
i = 0
j = np.flatnonzero(y_train != y_train[i])[0]

print("K(p,p) =", qek_similarity(P_train[i], P_train[i], mu=1.0))
print("K(p,q) =", qek_similarity(P_train[i], P_train[j], mu=1.0))

mus = np.array([0.5, 1, 2, 5, 10])

# TODO — compute the similarity for each value of mu
sims = [...]

plt.figure(figsize=(5.5, 3.5))
plt.plot(mus, sims, marker="o")
plt.xlabel(r"$\mu$")
plt.ylabel("QEK similarity")
plt.title("Effect of the kernel scale")
plt.grid(alpha=0.25)
plt.show()

**Short answer.** In one or two sentences, explain what happens when $\mu$ becomes very small or very large.

> TODO: write your answer here.


## Q3 — Build the kernel matrices

For a fixed value $\mu=1$, construct

- `K_train[i,j] = K(P_train[i], P_train[j])`;
- `K_test[i,j] = K(P_test[i], P_train[j])`.

Then check:
- the shapes of the two matrices;
- symmetry of `K_train`;
- unit diagonal;
- the smallest eigenvalue of the symmetrized training kernel matrix.


In [ ]:
def qek_matrix(A, B, mu=1.0):
    # TODO — allocate and fill the matrix
    K = np.empty((len(A), len(B)), dtype=float)

    for i, a in enumerate(A):
        for j, b in enumerate(B):
            K[i, j] = ...   # TODO

    return K


mu = 1.0

# TODO — build both kernel matrices
K_train = ...
K_test = ...

print("K_train shape:", K_train.shape)
print("K_test shape :", K_test.shape)

# TODO — verify the expected properties
print("Symmetric?    ", ...)
print("Unit diagonal?", ...)

K_sym = 0.5 * (K_train + K_train.T)
eigs = np.linalg.eigvalsh(K_sym)

print("Smallest eigenvalue:", eigs.min())
print("Largest eigenvalue :", eigs.max())

**Short answer.** What dimensions should `K_train` and `K_test` have, and why is the second matrix rectangular?

> TODO: write your answer here.


## Q4 — Visualize the kernel geometry

Display the training kernel matrix as a heat map. Then reorder the training samples **first by class label and then by graph size**, and display the reordered matrix.

Your figures should resemble the heat maps shown in the lab handout.


In [ ]:
# First heat map
plt.figure(figsize=(6, 5))

# TODO
plt.imshow(..., aspect="auto")

plt.colorbar(label="QEK similarity")
plt.xlabel("Training sample")
plt.ylabel("Training sample")
plt.title("QEK kernel matrix before reordering")
plt.show()


# Sort first by class, then by graph size.
# Hint: np.lexsort((secondary_key, primary_key))
order = ...   # TODO

K_sorted = K_train[np.ix_(..., ...)]   # TODO

plt.figure(figsize=(6, 5))
plt.imshow(K_sorted, aspect="auto")
plt.colorbar(label="QEK similarity")
plt.xlabel("Training sample (sorted)")
plt.ylabel("Training sample (sorted)")
plt.title("QEK kernel matrix, sorted by class then size")
plt.show()

**Short answer.** What structure becomes easier to see after reordering? Do you see effects associated with both class and graph size?

> TODO: write your answer here.


## Q5 — Train and evaluate an SVC

Train an SVC directly from the precomputed kernel matrices using

```python
SVC(kernel="precomputed", C=1.0)
```

Report:
- the confusion matrix;
- the F1 score;
- the number of support vectors.


In [ ]:
# TODO — instantiate and train the SVC
model = SVC(kernel="precomputed", C=...)
model.fit(..., ...)

# TODO — predict on the test set
y_pred = ...

# TODO — compute the requested metrics
cm = ...  # confusion matrix
f1 = ...  # F1-score

print(f"F1 score          : {f1:.3f}")
print("Support vectors per class:", model.n_support_)
print("Total support vectors:", len(model.support_))

ConfusionMatrixDisplay(cm).plot()
plt.title("QEK-SVC confusion matrix")
plt.show()


**Short answer.** What is the F1-score? What is the role of the support vectors in the SVC?

> TODO: write your answer here.


## Q6 — Size-only baseline

To test whether the classifier is exploiting mainly molecular size, compare QEK with the baseline kernel

$$K_{\text{size}}(N_i,N_j)
=
\exp[-\gamma(N_i-N_j)^2],
\qquad \gamma=0.5.$$

Complete the kernel-matrix function, train another SVC, and compare the two test performances.


In [ ]:
def size_kernel_matrix(sizes_A, sizes_B, gamma=0.5):
    a = np.asarray(sizes_A)[:, None]
    b = np.asarray(sizes_B)[None, :]

    # TODO
    return ...


# TODO — compute train/test kernel matrices
K_size_train = ...
K_size_test = ...

# TODO — fit the size-only SVC
size_model = SVC(kernel="precomputed", C=1.0)
size_model.fit(..., ...)

size_pred = ...

print(f"Size-kernel F1               : {f1_score(y_test, size_pred):.3f}")
print(f"QEK F1                       : {f1:.3f}")

**Short answer.** Is graph size alone predictive? Does it explain the entire QEK performance?

> TODO: write your answer here.
